In [ ]:
%run "/Users/manojrammopati/Public/Projects/bupa_insurance_project/_01_Bronze/Jupyter Notebooks/00_bronze_data_connector.ipynb"

Preconnectors ran successfully and connected to 'clientdatastorage' storage accounts
 'policy_df, claim_df, member_df, provider_df' are ready to use from container 'raw data'(meridian architecture of bronze) 


In [10]:
import sys
import importlib

# Add the Silver layer directory to sys.path BEFORE importing
silver_dir = "/Users/manojrammopati/Public/Projects/bupa_insurance_project/02_Silver_Layer"
if silver_dir not in sys.path:
    sys.path.insert(0, silver_dir)

# Now import (and reload to pick up edits)
import utils_silver
from utils_silver import *

importlib.reload(utils_silver)

paths_bronze, paths_silver = utils_silver.build_paths(storage_account1)
print(sorted(paths_silver.keys()))
print("Imported utils_silver from", utils_silver.__file__)

✅ utils_silver.py loaded
✅ utils_silver.py loaded
['_metrics', '_quarantine', '_ref_dim_channel', '_ref_dim_product_line', '_reference', 'claims', 'members', 'policies', 'providers']
Imported utils_silver from /Users/manojrammopati/Public/Projects/bupa_insurance_project/02_Silver_Layer/utils_silver.py


# 

# changing the gold data from silver to golddata 

In [ ]:
from pyspark.sql import functions as F

# Adjust if your account name is different
storage_account = "clientdatastorage"
GOLD_CONTAINER = "golddata"

GOLD_BASE = f"abfss://{GOLD_CONTAINER}@{storage_account}.dfs.core.windows.net"

fact_policies_path_new = f"{GOLD_BASE}/fact_policies"   # already used
fact_members_path_new  = f"{GOLD_BASE}/fact_members"
fact_claims_path_new   = f"{GOLD_BASE}/fact_claims"

print("GOLD_BASE:", GOLD_BASE)
print("fact_policies_path_new:", fact_policies_path_new)
print("fact_members_path_new :", fact_members_path_new)
print("fact_claims_path_new  :", fact_claims_path_new)


GOLD_BASE: abfss://golddata@clientdatastorage.dfs.core.windows.net
fact_policies_path_new: abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_policies
fact_members_path_new : abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_members
fact_claims_path_new  : abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_claims


In [ ]:
DB_GOLD = "bupa_gold"

# --- Read existing fact_members from the metastore ---
fact_members_df = spark.table(f"{DB_GOLD}.fact_members")
print("Existing fact_members rows:", fact_members_df.count())

# --- Write to the new Gold container path ---
(fact_members_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(fact_members_path_new))

print("✅ fact_members written to:", fact_members_path_new)

# --- Re-register the table to point at golddata ---
spark.sql(f"DROP TABLE IF EXISTS {DB_GOLD}.fact_members")
spark.sql(f"""
CREATE TABLE {DB_GOLD}.fact_members
USING DELTA
LOCATION '{fact_members_path_new}'
""")

print("✅ fact_members table now points to:", fact_members_path_new)

# Quick sanity check
spark.table(f"{DB_GOLD}.fact_members").show(5, truncate=False)


Existing fact_members rows: 381109


✅ fact_members written to: abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_members
✅ fact_members table now points to: abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_members
+------------+----------+---+------+------------------+-----------+---------------+-----------------+------+--------+----------+-------------+------------+-------------+----------------+--------------+------------+------------+
|Member_ID   |Policy_ID |Age|Gender|BMI               |Smoker_Flag|Chronic_Disease|Employment_Status|Region|Age_Band|BMI_Band  |Smoker_Status|Chronic_Flag|Chronic_Group|Employment_Group|Region_Group  |dq_age_valid|dq_bmi_valid|
+------------+----------+---+------+------------------+-----------+---------------+-----------------+------+--------+----------+-------------+------------+-------------+----------------+--------------+------------+------------+
|MEM_00000008|P_00000008|56 |F     |23.140940890221643|Y          |Asthma         |Student          |280   |45–59  

In [ ]:
# --- Read existing fact_claims from the metastore ---
fact_claims_df = spark.table(f"{DB_GOLD}.fact_claims")
print("Existing fact_claims rows:", fact_claims_df.count())

# --- Write to the new Gold container path ---
(fact_claims_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(fact_claims_path_new))

print("✅ fact_claims written to:", fact_claims_path_new)

# --- Re-register the table to point at golddata ---
spark.sql(f"DROP TABLE IF EXISTS {DB_GOLD}.fact_claims")
spark.sql(f"""
CREATE TABLE {DB_GOLD}.fact_claims
USING DELTA
LOCATION '{fact_claims_path_new}'
""")

print("✅ fact_claims table now points to:", fact_claims_path_new)

# Quick sanity check
spark.table(f"{DB_GOLD}.fact_claims").show(5, truncate=False)


Existing fact_claims rows: 558211


✅ fact_claims written to: abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_claims
✅ fact_claims table now points to: abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_claims


+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+-------------------+--------------+----------------------+--------------------+--------------+-------------+
|Claim_ID |Provider_ID|Member_Key|Date_Reported|Date_Settled|Payout_GBP|Claim_Amount_GBP  |Fraud_Label|Claim_Type|Claim_Status|Provider_Fraud_Flag|Days_To_Settle|Payout_to_Amount_Ratio|High_Cost_Claim_Flag|dq_money_valid|dq_date_valid|
+---------+-----------+----------+-------------+------------+----------+------------------+-----------+----------+------------+-------------------+--------------+----------------------+--------------------+--------------+-------------+
|CLM110013|PRV51790   |BENE15435 |NULL         |NULL        |200.0     |244.07511199775644|0          |Travel    |Settled     |0                  |NULL          |0.8194198841618822    |0                   |1             |1            |
|CLM110020|PRV53679   |BENE53030 |NULL         |NULL    

In [ ]:
for tbl in ["fact_policies", "fact_members", "fact_claims"]:
    print(f"\n=== {DB_GOLD}.{tbl} ===")
    loc = spark.sql(f"DESCRIBE DETAIL {DB_GOLD}.{tbl}").select("location").first()[0]
    print("Location:", loc)



=== bupa_gold.fact_policies ===
Location: abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_policies

=== bupa_gold.fact_members ===
Location: abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_policies

=== bupa_gold.fact_members ===
Location: abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_members

=== bupa_gold.fact_claims ===
Location: abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_members

=== bupa_gold.fact_claims ===
Location: abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_claims
Location: abfss://golddata@clientdatastorage.dfs.core.windows.net/fact_claims
